# 01 — PALM designs

Writes one directory per analysis (`output/voxel_statistics/palm/<family>/<domain>/`) holding
`design.csv`, `contrast.csv`, `input_paths.txt` and, for case-control designs, `eb.csv`.

| family | inputs | design | inference |
|---|---|---|---|
| `lnm_nocov` (primary) | lesion-derived connectivity maps | impaired vs unimpaired + cohort | label permutation within cohort |
| `lnm_volnihss` | connectivity maps | + lesion volume + NIHSS | label permutation within cohort |
| `vlsm_nocov`, `vlsm_volnihss` | binary lesion masks | as the two LNM models | label permutation within cohort |
| `lnm_xdomcov` | connectivity maps | + impairment labels of the two other domains (attention/executive, language, verbal memory) | label permutation within cohort |
| `onesample_cohort`, `onesample_volnihss` | connectivity maps of impaired patients | intercept + demeaned cohort (+ volume + NIHSS) | sign flipping |

In [ ]:
import sys; sys.path.insert(0, "utils")
from config import *

clinical = pd.read_csv(CLINICAL_CSV).dropna(subset=["sub_id"])
COHORT, NIHSS = "demographics-cohort", "stroke-NIHSS"
VOL = "stroke-total_infarct_volume" if "stroke-total_infarct_volume" in clinical else "stroke-total_infarct_volume_ml"

clinical = clinical.copy()
clinical["sub_id"] = clinical["sub_id"].astype(str).str.replace(r"^(CAS|USCOG)(\d+)$", r"\1_\2", regex=True)
for c in [VOL, NIHSS] + KEYS:
    clinical[c] = pd.to_numeric(clinical[c], errors="coerce")

# NIHSS: cohort-median imputation, global median for cohorts without any NIHSS
clinical["nihss_imp"] = (clinical[NIHSS].fillna(clinical.groupby(COHORT)[NIHSS].transform("median"))
                         .fillna(clinical[NIHSS].median()))
clinical["intercept"] = 1
print(len(clinical), "patients |", int(clinical[NIHSS].isna().sum()), "missing NIHSS values imputed")

def flnm_path(s): return f"{FLNM_DIR}/{s}_infarct_conn-GSP1000v2xFullSet_desc-T_funcmap.nii.gz"
def lesion_path(s): return f"{LESION_DIR}/{s}_infarct.nii.gz"

In [ ]:
manifest = []

def write_variant(family, key, sub, design_cols, image_fn, eb=None):
    out = PALM_DIR / family / key
    out.mkdir(parents=True, exist_ok=True)
    sub[design_cols].to_csv(out / "design.csv", header=False, index=False)
    pd.DataFrame([[1] + [0] * (len(design_cols) - 1)]).to_csv(out / "contrast.csv", header=False, index=False)
    paths = sub["sub_id"].map(image_fn)
    paths.to_csv(out / "input_paths.txt", header=False, index=False)
    if eb is not None:
        pd.DataFrame({"eb": eb}).to_csv(out / "eb.csv", header=False, index=False)
    manifest.append({"variant": f"{family}/{key}", "n": len(sub), "n_cols": len(design_cols),
                     "n_cohorts": sub[COHORT].nunique(), "missing_images": int((~paths.map(lambda p: Path(p).exists())).sum())})

def cohort_dummies(sub):
    d = pd.get_dummies(sub[COHORT], prefix="cohort", drop_first=True).astype(int)
    return pd.concat([sub, d], axis=1), list(d.columns)

def case_control(family, image_fn, extra, keys=KEYS, other_labels=False):
    for k in keys:
        cov = [o for o in keys if o != k] if other_labels else extra
        sub = clinical.dropna(subset=["sub_id", k, COHORT] + cov).copy()
        labels = [k] + [c for c in cov if c in KEYS]
        sub[labels] = sub[labels].astype(int)
        sub, coh = cohort_dummies(sub)
        write_variant(family, k, sub, [k] + cov + coh + ["intercept"], image_fn,
                      eb=pd.factorize(sub[COHORT])[0] + 1)      # permute labels within cohort

def one_sample(family, extra):
    for k in KEYS:
        sub = clinical[clinical[k] == 1].dropna(subset=["sub_id", COHORT] + extra).copy()
        sub, coh = cohort_dummies(sub)
        for c in extra + coh:                                      # demeaned: intercept = adjusted mean
            sub[f"dm_{c}"] = sub[c] - sub[c].mean()
        write_variant(family, k, sub, ["intercept"] + [f"dm_{c}" for c in extra + coh], flnm_path)

case_control(LNM, flnm_path, [])
case_control(LNM_FULL, flnm_path, [VOL, "nihss_imp"])
case_control(VLSM, lesion_path, [])
case_control(VLSM_FULL, lesion_path, [VOL, "nihss_imp"])
case_control(LNM_XDOM, flnm_path, [], keys=XDOM_KEYS, other_labels=True)
one_sample(OS, [])
one_sample(OS_FULL, [VOL, "nihss_imp"])

manifest = pd.DataFrame(manifest)
manifest.to_csv(PALM_DIR / "_manifest.csv", index=False)
assert (manifest["missing_images"] == 0).all()
manifest